<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/Topics/Working_with_Tokenizers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!gdown --fuzzy https://drive.google.com/file/d/1tRrkTL_mo3fqrcppHHOd-y4JMIHh-opz/view?usp=sharing -O comments.json.gz

Downloading...
From (original): https://drive.google.com/uc?id=1tRrkTL_mo3fqrcppHHOd-y4JMIHh-opz
From (redirected): https://drive.google.com/uc?id=1tRrkTL_mo3fqrcppHHOd-y4JMIHh-opz&confirm=t&uuid=4abbe5b8-9d8b-446e-8b78-2a2d337fb46f
To: /content/comments.json.gz
100% 106M/106M [00:04<00:00, 24.2MB/s]


In [5]:
!gunzip comments.json.gz

gzip: comments.json already exists; do you wish to overwrite (y or n)? n
	not overwritten


In [6]:
import pandas as pd
import json

with open('comments.json', 'r') as fp:
    texts = json.load(fp)

texts = [line['content'] for line in texts]
texts = pd.Series(texts)
texts = texts[texts.notna()]

In [7]:
texts = texts.sample(10_000, random_state=42)

In [8]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

In [9]:
def train_tokenizer(text):
  special_tokens = ['[UNK]', '[PAD]', '[UNK]', 'MASK']
  tokenizer = Tokenizer(models.WordPiece(unk_token = special_tokens[2]))
  tokenizer.normalizer = normalizers.Sequence([
      normalizers.NFD(),
      normalizers.Lowercase(),
      normalizers.StripAccents()
  ])

  trainer = trainers.WordPieceTrainer(
      vocab_size = 10000,
      special_tokens = special_tokens
  )

  tokenizer.train_from_iterator(text, trainer = trainer)
  tokenizer.save("tokenizer.json")
  return tokenizer


In [10]:
%%time

tokenizer = train_tokenizer(texts)

CPU times: user 13.5 s, sys: 579 ms, total: 14.1 s
Wall time: 9.42 s


In [11]:
tokenizer.encode("трансформер").tokens

['т', '##ран', '##с', '##фор', '##мер']

In [12]:
from collections import Counter


def get_top_k_items(tokenizer, texts, k):
  token_counts = Counter()
  batch_size = 1000
  for i in range(0, len(texts), batch_size):
    batch = texts[i: i+batch_size]target_model
    encodings = tokenizer.encode_batch(batch.tolist())
    for encoding in encodings:
      tokens = encoding.tokens
      token_counts.update(tokens)
  top_k = [token for token, count in token_counts.most_common(k)]
  return top_k



In [13]:
top_k = get_top_k_items(tokenizer, texts, 1000)
top_k[0:10]

['[UNK]',
 '##.',
 '##!',
 '## ',
 '##и ',
 '##удал',
 '##не ',
 'отзыв ',
 '##ен ',
 '##в ']

In [19]:
from transformers import BertTokenizer, BertForMaskedLM

model_name = 'bert-base-multilingual-cased'

In [20]:
source_tokenizer = BertTokenizer.from_pretrained(model_name)
source_model = BertForMaskedLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
target_tokenizer = BertTokenizer.from_pretrained(model_name)
target_model = BertForMaskedLM.from_pretrained(model_name)
target_tokenizer.add_tokens(top_k)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1000

In [22]:
source_tokenizer.get_vocab()

{'##eku': 41027,
 '⇔': 1798,
 '##ציות': 67258,
 'específico': 79336,
 'Imre': 45828,
 'Yunani': 35952,
 '##叠': 112509,
 '궐': 8918,
 '##бка': 59327,
 '##渉': 114830,
 '##primerie': 90575,
 'Nación': 63060,
 '##ladi': 28645,
 '##чни': 21674,
 '##芦': 116451,
 '##вная': 67869,
 'Atelier': 61026,
 'высота': 86399,
 'Петра': 34634,
 '##土': 112794,
 '##ানী': 97671,
 '##krát': 34339,
 'tué': 53758,
 'versos': 78420,
 'excess': 93317,
 '橇': 4735,
 '##חיל': 49132,
 'Ordine': 54428,
 'Bobby': 19371,
 'katika': 13743,
 'decorative': 103374,
 'titre': 13777,
 'कोड': 59850,
 '##ྫ': 111468,
 '##भा': 104741,
 '##ဉ': 111478,
 'cinci': 79300,
 'começou': 25088,
 'augustus': 19947,
 '##ئد': 50933,
 '##द': 15552,
 'született': 26822,
 'Iván': 57343,
 '##솟': 119056,
 'Lightning': 47193,
 '##nské': 53111,
 '##こと': 28100,
 '##alar': 46988,
 'dok': 17674,
 '穰': 6001,
 '1539': 54103,
 'کنار': 49430,
 '##uang': 43383,
 '##eña': 74008,
 'Lê': 33457,
 'turneringen': 81347,
 'managing': 61274,
 'blisko': 88670,
 '#

In [23]:
source_model

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

In [32]:
import torch
def get_new_embeddings(source_model, source_tokenizer, target_tokenizer, top_k):
  source_vocab = source_tokenizer.get_vocab()
  target_vocab = target_tokenizer.get_vocab()
  target_inverted_vocab = {v:i for i, v in target_vocab.items()}


  source_input_embeddings = source_model.get_input_embeddings()
  input_embeddings = source_input_embeddings.weight.data
  new_embeddings = []

  for i in range(len(source_vocab), len(target_vocab)):
    target_token = target_inverted_vocab[i]

    if target_token not in top_k:
      continue
    tokenized = source_tokenizer.tokenize(target_token)
    tokenized_ids = source_tokenizer.convert_tokens_to_ids(tokenized)


    subtoken_embeddings = input_embeddings[torch.tensor(tokenized_ids)]
    mean_embeddings = subtoken_embeddings.mean(dim = 0)
    new_embeddings.append(mean_embeddings)

  new_embeddings = torch.stack(new_embeddings, dim = 0)
  new_embeddings = torch.cat([input_embeddings, new_embeddings], dim = 0)
  return new_embeddings


In [33]:
new_embeddings = get_new_embeddings(source_model, source_tokenizer, target_tokenizer, top_k)

In [34]:
new_embeddings.shape

torch.Size([120369, 768])

In [35]:
len(target_tokenizer)

120369

In [36]:
len(source_tokenizer)

119547

In [37]:
target_model.resize_token_embeddings(len(target_tokenizer))

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(120369, 768, padding_idx=0)

In [38]:
target_model.get_input_embeddings().weight_data = new_embeddings

In [44]:
target_model.bert.embeddings.word_embeddings.weight.data.shape

torch.Size([120369, 768])

In [45]:
new_embeddings.shape

torch.Size([120369, 768])

In [46]:
target_model.bert.embeddings.word_embeddings.weight.data = new_embeddings